<a href="https://colab.research.google.com/github/PaulofSticker/-PaulofSticker-exercicio_gulp./blob/main/Week6_Lab2_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 6 — Lab 2: Column-level encryption
In this lab, I created a SQLite database with five fictional patients and encrypted the SSN and diagnosis fields using AES-GCM. I used a data encryption key (DEK), protected by a key encryption key (KEK), to demonstrate envelope encryption. I compared decrypted results with raw database records, rotated the DEK, and tested whether tampered data was rejected. The KEK remained in notebook memory, so this exercise simulated key management rather than using a real KMS.


In [1]:
import os, sqlite3
from pathlib import Path
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.exceptions import InvalidTag

KEK = AESGCM.generate_key(bit_length=256)
print("DEMO KEK (hex, printed once):", KEK.hex())
DEK = AESGCM.generate_key(bit_length=256)
used_nonces = set()

def seal(key, value, aad):
    nonce = os.urandom(12)
    while (key, nonce) in used_nonces:
        nonce = os.urandom(12)
    used_nonces.add((key, nonce))
    return nonce + AESGCM(key).encrypt(nonce, value, aad)

def unseal(key, blob, aad):
    return AESGCM(key).decrypt(blob[:12], blob[12:], aad)

wrapped_dek = seal(KEK, DEK, b"patients:dek:v1")
assert unseal(KEK, wrapped_dek, b"patients:dek:v1") == DEK
print("DEK wrapped successfully; plaintext DEK not printed.")


DEMO KEK (hex, printed once): 0ec3190127558cc50f1f72c64ba838fbbd8e0b44828f85a79a38b26b69672fa9
DEK wrapped successfully; plaintext DEK not printed.


In [2]:
db_path = "week6_patients.db"
conn = sqlite3.connect(db_path)
conn.execute("DROP TABLE IF EXISTS patients")
conn.execute("CREATE TABLE patients(id INTEGER PRIMARY KEY, name TEXT, ssn_enc BLOB, diagnosis_enc BLOB)")
patients = [
    (1, "Demo Patient A", "000-00-0001", "Demo diagnosis A"),
    (2, "Demo Patient B", "000-00-0002", "Demo diagnosis B"),
    (3, "Demo Patient C", "000-00-0003", "Demo diagnosis C"),
    (4, "Demo Patient D", "000-00-0004", "Demo diagnosis D"),
    (5, "Demo Patient E", "000-00-0005", "Demo diagnosis E"),
]
def aad_for(pid, field):
    return f"patients:{pid}:{field}".encode()

with conn:
    for pid, name, ssn, diagnosis in patients:
        conn.execute("INSERT INTO patients VALUES (?, ?, ?, ?)", (
            pid, name,
            seal(DEK, ssn.encode(), aad_for(pid, "ssn")),
            seal(DEK, diagnosis.encode(), aad_for(pid, "diagnosis"))))
print("Inserted 5 fictional patients; fresh nonce per encrypted value.")


Inserted 5 fictional patients; fresh nonce per encrypted value.


In [3]:
def decrypt_and_print(wrapped, version):
    app_dek = unseal(KEK, wrapped, f"patients:dek:v{version}".encode())
    result = []
    print("APPLICATION VIEW: id | name | ssn | diagnosis")
    for pid, name, ssn_blob, diagnosis_blob in conn.execute("SELECT * FROM patients ORDER BY id"):
        row = (pid, name,
               unseal(app_dek, ssn_blob, aad_for(pid, "ssn")).decode(),
               unseal(app_dek, diagnosis_blob, aad_for(pid, "diagnosis")).decode())
        result.append(row)
        print(row)
    return result

before = decrypt_and_print(wrapped_dek, 1)
assert before == patients


APPLICATION VIEW: id | name | ssn | diagnosis
(1, 'Demo Patient A', '000-00-0001', 'Demo diagnosis A')
(2, 'Demo Patient B', '000-00-0002', 'Demo diagnosis B')
(3, 'Demo Patient C', '000-00-0003', 'Demo diagnosis C')
(4, 'Demo Patient D', '000-00-0004', 'Demo diagnosis D')
(5, 'Demo Patient E', '000-00-0005', 'Demo diagnosis E')


In [4]:
conn.commit()
print("RAW SELECT * FROM patients — reopened DB without decryption")
with sqlite3.connect(db_path) as raw:
    for row in raw.execute("SELECT * FROM patients ORDER BY id"):
        print(row)
        assert isinstance(row[2], bytes) and isinstance(row[3], bytes)
raw_bytes = Path(db_path).read_bytes()
assert all(ssn.encode() not in raw_bytes and diagnosis.encode() not in raw_bytes
           for _, _, ssn, diagnosis in patients)
print("Verified: sensitive plaintext strings absent from saved file.")
print("Names and IDs are deliberately plaintext.")


RAW SELECT * FROM patients — reopened DB without decryption
(1, 'Demo Patient A', b'\xbf\xb0\x8c}Or\x9e\xfa\x079Z\x152{xx\xbf&\x13\x14\xe0\xacjZ\x02\xc5\xd2\xf9\xff\xb2\xae^\x96\x94\xb0$E]+', b'm\xd4\xe0\xa0\x8b\xcb\x9c\x94\x9e{\xca\x99\x829\x81},\x1b\xfc\xea\xa1\xe3\xc3/ \xd7\x7fh\xd5\xae1\xac\xc0\x7f0/W\xad\xf9\t\\\xe7#/')
(2, 'Demo Patient B', b'\xc05\x97\xf6p(\xc7\x84\xda\xe7b\x9f\x9e\xc0j\r\x03\x1e\x82\x18H\xaf!g\xe2\xafS\x8f\xd3\xe5\x03\xf0\x94\r\xb8\xd7?vq', b'\xfd\xc3\x98\tA\xfb{Eo\tgW\x81\xe9ZO\xc6]\xbf\x88\xe2B\x06/\xff\x8a\x8c\x87\x1a/%)?\x03\xc3\xa7{\xe6\xb7\xd0fM7\xef')
(3, 'Demo Patient C', b'X\xa2\xbd\x8d\xca\xda\x03\xa6K\x83\xf8\x88\xf2\n\xd8,\xe3\xa0\x03r\x00-\xfa\xadV\xf8\x16\xf6|\xd0n\xba\xa7\xd8U/\x7f\x972', b':\xb9\x86-Q\xc6x\xeeSuF\xfb\x80\xa3\x14\x0b"X\xdb\x9c\x8b\xf3\x02-\x01\\)\xbb\xea\xa3\x7f\xe0\x12!\xc7bD\x1e\x84tC\t?\x0e')
(4, 'Demo Patient D', b'\x8a\xca\xc4{\xd1p\x13\x05\x11a\xdd\xeb\xdf\xd0l\xe1\xde\xb0\xc3\xcf-fuM\xff\xd4\xab\xad\xbc\xea\xba\xaa\xe1\xc8

In [5]:
# DEK rotation: re-encrypt every sensitive field and wrap the new DEK.
old_dek = unseal(KEK, wrapped_dek, b"patients:dek:v1")
new_dek = AESGCM.generate_key(bit_length=256)
new_wrapped_dek = seal(KEK, new_dek, b"patients:dek:v2")
old_blobs = list(conn.execute("SELECT ssn_enc, diagnosis_enc FROM patients ORDER BY id"))
with conn:
    for pid, name, ssn_blob, diagnosis_blob in list(conn.execute("SELECT * FROM patients")):
        ssn = unseal(old_dek, ssn_blob, aad_for(pid, "ssn"))
        diagnosis = unseal(old_dek, diagnosis_blob, aad_for(pid, "diagnosis"))
        conn.execute("UPDATE patients SET ssn_enc=?, diagnosis_enc=? WHERE id=?", (
            seal(new_dek, ssn, aad_for(pid, "ssn")),
            seal(new_dek, diagnosis, aad_for(pid, "diagnosis")), pid))
wrapped_dek = new_wrapped_dek
DEK = new_dek
after = decrypt_and_print(wrapped_dek, 2)
assert after == before
new_blobs = list(conn.execute("SELECT ssn_enc, diagnosis_enc FROM patients ORDER BY id"))
assert all(a != b for a, b in zip(old_blobs, new_blobs))
print("ROTATION PASSED: plaintext unchanged; ciphertext changed.")
# Deleting Python references does not guarantee secure erasure of memory.
del old_dek


APPLICATION VIEW: id | name | ssn | diagnosis
(1, 'Demo Patient A', '000-00-0001', 'Demo diagnosis A')
(2, 'Demo Patient B', '000-00-0002', 'Demo diagnosis B')
(3, 'Demo Patient C', '000-00-0003', 'Demo diagnosis C')
(4, 'Demo Patient D', '000-00-0004', 'Demo diagnosis D')
(5, 'Demo Patient E', '000-00-0005', 'Demo diagnosis E')
ROTATION PASSED: plaintext unchanged; ciphertext changed.


In [6]:
# Optional integrity test: a changed byte must fail authentication.
blob = conn.execute("SELECT ssn_enc FROM patients WHERE id=1").fetchone()[0]
changed = blob[:-1] + bytes([blob[-1] ^ 1])
try:
    unseal(DEK, changed, aad_for(1, "ssn"))
except InvalidTag:
    print("PASS: tampered value rejected with InvalidTag.")
else:
    raise AssertionError("Tampering unexpectedly accepted")


PASS: tampered value rejected with InvalidTag.


In [7]:
conn.commit()
try:
    from google.colab import files
    files.download(db_path)
except ImportError:
    print("Run this cell in Colab to download:", db_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Reflection.
I used Google Colab to create an SQLite database with five fictitious patients. My goal was to protect the SSN and diagnosis fields before storing the information. To do this, I used AES-GCM and two keys: the DEK, responsible for encrypting the data, and the KEK, responsible for protecting the DEK.
I learned that this process is called envelope encryption. I understood that separating the functions of the keys helps to organize data protection. However, in this exercise, the KEK remained in the notebook's memory. Therefore, the lab simulated the operation of a key management service, without using a real KMS.
When executing the decryption function, I was able to view the patients' information. Then, I queried the database directly, without decrypting. The names and identifiers remained readable, while SSN and diagnosis appeared as byte sequences. This comparison showed that the database stores the sensitive fields encrypted, but does not protect all the information in the table.
I also rotated the DEK. The program created a new key, decrypted the existing values, and re-encrypted the sensitive fields. The confirmation message indicated that the information was preserved, while the encrypted values ​​changed. I learned that this rotation requires updating the data, not just swapping a variable.
In the integrity test, the code altered a byte of an encrypted value and checked if the decryption rejected this change. This test demonstrates the importance of authentication to identify tampered data.
I concluded that this scheme protects sensitive fields when someone obtains only the file from the database, without access to the keys. However, it does not prevent a compromised application from reading the data, nor does it protect information already decrypted in memory. I also realized that names and identifiers can reveal information. Therefore, I need to combine cryptography with access control and secure key management.
